# Indirect GRPO Training Pipeline

**Bradley-Terry Reward Model → GRPO fine-tuning of Qwen 2.5 7B Instruct**

| Step | What happens | Runtime (A100) |
|------|-------------|----------------|
| 1 | Clone repo + install deps | ~2 min |
| 2 | Train BT reward model (Qwen 1.5B) | ~25–40 min |
| 3 | Evaluate RM accuracy on held-out pairs | ~1 min |
| 4 | Indirect GRPO training (Qwen 7B + LoRA) | ~45–60 min |
| 5 | Save checkpoints to Google Drive | ~2 min |

> **Priority**: GRPO checkpoint saved every 50 steps. Full head-to-head eval vs base model is skipped here — run separately once you have the checkpoint.

In [ ]:
# Optional: mount Google Drive so models persist after session ends
# Skip this cell only if you plan to download the zips manually at the end
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_SAVE_PATH = "/content/drive/MyDrive/cs234_indirect_grpo"
    import os; os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    print(f"Drive mounted. Models will be saved to: {DRIVE_SAVE_PATH}")
except Exception as e:
    DRIVE_SAVE_PATH = None
    print(f"Drive not mounted ({e})\nModels will be available for download at the end.")

In [ ]:
# Clone repo (or pull latest if already cloned)
!git clone https://github.com/zcsabbagh/cs234-project.git /content/cs234-project 2>/dev/null \
  || git -C /content/cs234-project pull

%cd /content/cs234-project

# Install dependencies
!pip install -q \
    torch \
    "transformers>=5.2.0" \
    "trl>=0.28.0" \
    "peft>=0.15.0" \
    "accelerate>=1.12.0" \
    "datasets>=4.5.0" \
    "openai>=2.24.0"

print("\nInstallation complete.")
print("If this is your FIRST run: Runtime -> Restart runtime, then re-run from this cell.")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Enable GPU: Runtime -> Change runtime type -> GPU (A100 recommended)")

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU:  {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GB")
print(f"BF16: {torch.cuda.is_bf16_supported()}")

# Auto-set batch sizes based on VRAM
if gpu_mem_gb >= 70:       # H100 / A100-80GB
    RM_BATCH = 16; GRPO_BATCH = 2; GRPO_G = 8
elif gpu_mem_gb >= 35:     # A100-40GB
    RM_BATCH = 8;  GRPO_BATCH = 2; GRPO_G = 8
elif gpu_mem_gb >= 20:     # A30 / RTX 3090
    RM_BATCH = 4;  GRPO_BATCH = 1; GRPO_G = 4
else:                      # T4 (16 GB)
    RM_BATCH = 2;  GRPO_BATCH = 1; GRPO_G = 4

print(f"\nAuto-configured:")
print(f"  RM:   batch_size={RM_BATCH}")
print(f"  GRPO: batch_size={GRPO_BATCH}, G={GRPO_G} (completions/prompt)")

In [ ]:
import os

REPO_DIR    = "/content/cs234-project"
DATA_PATH   = f"{REPO_DIR}/preferences.jsonl"
RM_OUTPUT   = f"{REPO_DIR}/reward_model"
GRPO_OUTPUT = f"{REPO_DIR}/grpo_indirect"

# Fallback in case the Drive mount cell was skipped
if "DRIVE_SAVE_PATH" not in dir():
    DRIVE_SAVE_PATH = None

# Together API key — only needed for final eval (we skip it here)
# Add via Colab Secrets (key icon in sidebar) with name TOGETHER_API_KEY
try:
    from google.colab import userdata
    os.environ["TOGETHER_API_KEY"] = userdata.get("TOGETHER_API_KEY")
    print("TOGETHER_API_KEY loaded from Colab secrets")
except Exception:
    os.environ.setdefault("TOGETHER_API_KEY", "")
    print("TOGETHER_API_KEY not set (fine for training, needed for final eval later)")

# Verify scripts and data are present
for fname in ["train_reward_model.py", "train_grpo_indirect_eval.py", "preferences.jsonl"]:
    path = f"{REPO_DIR}/{fname}"
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"  {fname}  ({size:,} bytes)")
    else:
        print(f"  MISSING: {fname}")
        if fname == "train_grpo_indirect_eval.py":
            print("    -> Push this file to GitHub first: git add train_grpo_indirect_eval.py && git push")

n_records = sum(1 for _ in open(DATA_PATH))
print(f"\npreferences.jsonl: {n_records:,} preference records")

---
## Step 1: Train Bradley-Terry Reward Model

- **Base model**: `Qwen/Qwen2.5-1.5B-Instruct` (small enough to train fast, big enough to judge quality)
- **Data**: winner/loser pairs from `preferences.jsonl` (labeled by Llama 70B)
- **Loss**: $\mathcal{L} = -\log\sigma\!\left(r_\theta(x, y_w) - r_\theta(x, y_l)\right)$
- **Saves**: best checkpoint to `./reward_model/` (evaluated every 300 steps by eval loss)
- **Max sequence length**: 512 tokens (reduced from 1024 default to save VRAM)

Training takes ~**25–40 min** on A100, ~**60 min** on T4.

In [ ]:
!python {REPO_DIR}/train_reward_model.py \
    --data {DATA_PATH} \
    --output {RM_OUTPUT} \
    --batch-size {RM_BATCH} \
    --grad-accum 4 \
    --epochs 3 \
    --lr 5e-5 \
    --max-length 512

---
## Step 2: Evaluate Reward Model

Accuracy on held-out preference pairs tells us how noisy the reward signal will be during GRPO:

| Accuracy | Interpretation | Action |
|----------|---------------|--------|
| ≥ 70% | Strong RM | Proceed with G=4 |
| 65–70% | Acceptable (~our case: 69%) | Use G=8 to average noise |
| < 65% | Weak RM | Improve training data first |

In [ ]:
import json, random, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print(f"Loading RM from {RM_OUTPUT} ...")
rm_tok = AutoTokenizer.from_pretrained(RM_OUTPUT)
if rm_tok.pad_token is None:
    rm_tok.pad_token = rm_tok.eos_token

device = torch.device("cuda")
dtype  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

rm_model = AutoModelForSequenceClassification.from_pretrained(
    RM_OUTPUT, num_labels=1, torch_dtype=dtype
).to(device).eval()
print("RM loaded.")

# ── Accuracy on held-out pairs ────────────────────────────────────────────
random.seed(42)
pairs = []
with open(DATA_PATH) as f:
    for line in f:
        row = json.loads(line)
        if "winner" in row and "s1" in row and "s2" in row:
            winner_text = row[row["winner"]]
            loser_key   = "s1" if row["winner"] == "s2" else "s2"
            pairs.append((row["instruction"], winner_text, row[loser_key]))

eval_pairs = random.sample(pairs, min(500, len(pairs)))
print(f"Evaluating on {len(eval_pairs)} held-out pairs ...")

correct = 0
for instr, winner, loser in eval_pairs:
    messages = [{"role": "user", "content": instr}]
    prompt   = rm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = rm_tok(
        [prompt + winner, prompt + loser],
        return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        scores = rm_model(**inputs).logits[:, 0].tolist()
    if scores[0] > scores[1]:
        correct += 1

acc = correct / len(eval_pairs)
print(f"\nRM Accuracy: {acc:.1%}  ({correct}/{len(eval_pairs)})")
if acc >= 0.70:
    print("Strong RM — GRPO rewards will be clean")
elif acc >= 0.65:
    print("Acceptable RM — using G=8 in GRPO to average out noise")
else:
    print("Weak RM — consider improving training data before GRPO")

# ── Sanity check ──────────────────────────────────────────────────────────
print("\nSanity check (good vs bad response):")
messages = [{"role": "user", "content": "What is machine learning?"}]
prompt = rm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

good = (
    "Machine learning is a subset of artificial intelligence where systems learn "
    "patterns from data to make predictions without being explicitly programmed."
)
bad = "idk lol"

inputs = rm_tok(
    [prompt + good, prompt + bad],
    return_tensors="pt", truncation=True, max_length=512, padding=True
).to(device)
with torch.no_grad():
    scores = rm_model(**inputs).logits[:, 0].tolist()

print(f"  Good response score: {scores[0]:.4f}")
print(f"  Bad response score:  {scores[1]:.4f}")
print(f"  Result: {'PASSED' if scores[0] > scores[1] else 'FAILED — check RM tokenizer format'}")

# Free RM from VRAM before loading 7B policy
del rm_model
torch.cuda.empty_cache()
print("\nRM freed from VRAM — ready for GRPO.")

---
## Step 3: Indirect GRPO Training

- **Policy**: `Qwen/Qwen2.5-7B-Instruct` fine-tuned with LoRA (r=16, α=64, targets q/k/v/o projections)
- **Reward**: BT reward model inference — fast local forward pass, zero API calls during training
- **G**: 8 completions/prompt (increased from 4 to average out noise from ~69% RM accuracy)
- **KL penalty**: β=0.05 — keeps policy close to reference Qwen 7B, reduces reward hacking
- **Checkpoints**: saved every 50 steps → `./grpo_indirect/checkpoint-N/`

**Quick eval at each checkpoint** (no API, ~30s):
- RM score on 8 freshly-generated completions (mean ± std)
- Response length mean — rising length + flat RM score signals padding hack
- Type-token ratio — falling TTR signals format collapse

Full head-to-head eval vs base model: run later with `--skip-final-eval` removed.

In [ ]:
!python {REPO_DIR}/train_grpo_indirect_eval.py \
    --reward-model {RM_OUTPUT} \
    --data {DATA_PATH} \
    --output {GRPO_OUTPUT} \
    --num-generations {GRPO_G} \
    --batch-size {GRPO_BATCH} \
    --max-steps 200 \
    --checkpoint-eval-prompts 8 \
    --skip-final-eval

---
## Step 4: Results & Save Checkpoint

Quick diagnostic: check if RM score improved over training and whether any reward-hacking signatures appeared.

In [ ]:
import json, os

summary_path = f"{GRPO_OUTPUT}/checkpoint_eval_summary.json"

if not os.path.exists(summary_path):
    print(f"No summary found at {summary_path}")
    print("Training may not have completed or no checkpoints were saved.")
else:
    with open(summary_path) as f:
        results = json.load(f)

    print(f"{'Step':>6}  {'RM Score':>10}  {'±Std':>7}  {'Len (chars)':>12}  {'TTR':>6}")
    print("─" * 52)
    for r in results:
        print(
            f"{r['step']:>6}  "
            f"{r['rm_score_mean']:>10.4f}  "
            f"{r['rm_score_std']:>7.4f}  "
            f"{r['response_length_mean']:>12.1f}  "
            f"{r['type_token_ratio']:>6.3f}"
        )

    if len(results) >= 2:
        rm_d  = results[-1]["rm_score_mean"]       - results[0]["rm_score_mean"]
        len_d = results[-1]["response_length_mean"] - results[0]["response_length_mean"]
        ttr_d = results[-1]["type_token_ratio"]     - results[0]["type_token_ratio"]
        print(f"\nTrends (end - start):")
        print(f"  RM score:   {rm_d:+.4f}  {'improving' if rm_d > 0 else 'degrading'}")
        print(f"  Resp len:   {len_d:+.1f}  {'watch for padding hack' if len_d > 50 else 'stable'}")
        print(f"  TTR:        {ttr_d:+.4f}  {'diversity dropping — possible collapse' if ttr_d < -0.05 else 'stable'}")

# List saved checkpoints
print(f"\nSaved checkpoints in {GRPO_OUTPUT}:")
if os.path.exists(GRPO_OUTPUT):
    ckpts = sorted(
        [d for d in os.listdir(GRPO_OUTPUT) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1])
    )
    for ckpt in ckpts:
        ckpt_dir = os.path.join(GRPO_OUTPUT, ckpt)
        size_mb  = sum(
            os.path.getsize(os.path.join(ckpt_dir, f))
            for f in os.listdir(ckpt_dir)
            if os.path.isfile(os.path.join(ckpt_dir, f))
        ) / 1e6
        print(f"  {ckpt}  ({size_mb:.0f} MB)")
    if not ckpts:
        print("  (no checkpoint subdirectories found)")
else:
    print(f"  Output directory {GRPO_OUTPUT} does not exist yet.")

In [ ]:
import shutil, os

if DRIVE_SAVE_PATH:
    # ── Copy to Google Drive ─────────────────────────────────────────────
    print(f"Saving models to Google Drive: {DRIVE_SAVE_PATH}")

    for name, src in [("grpo_indirect", GRPO_OUTPUT), ("reward_model", RM_OUTPUT)]:
        if not os.path.exists(src):
            print(f"  SKIP {name}: source directory not found")
            continue
        dst = f"{DRIVE_SAVE_PATH}/{name}"
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        size_gb = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(dst)
            for f in files
        ) / 1e9
        print(f"  {name} -> {dst}  ({size_gb:.2f} GB)")

    print("\nDone! To reload the GRPO model later:")
    print("  from peft import PeftModel")
    print("  from transformers import AutoModelForCausalLM, AutoTokenizer")
    print("  import torch")
    print(f"  base = AutoModelForCausalLM.from_pretrained(")
    print(f"      'Qwen/Qwen2.5-7B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')")
    print(f"  model = PeftModel.from_pretrained(base, '{DRIVE_SAVE_PATH}/grpo_indirect')")

else:
    # ── Download as zip ──────────────────────────────────────────────────
    print("Drive not mounted. Creating zip archives for download...")
    REPO_DIR = "/content/cs234-project"

    for name, src_dir in [("grpo_indirect", GRPO_OUTPUT), ("reward_model", RM_OUTPUT)]:
        if not os.path.exists(src_dir):
            print(f"  SKIP {name}: not found")
            continue
        zip_path = f"/content/{name}"
        shutil.make_archive(zip_path, "zip", REPO_DIR, name)
        size_mb = os.path.getsize(zip_path + ".zip") / 1e6
        print(f"  Created {zip_path}.zip  ({size_mb:.0f} MB)")

    from google.colab import files
    print("\nStarting downloads...")
    for name in ["grpo_indirect", "reward_model"]:
        zip_path = f"/content/{name}.zip"
        if os.path.exists(zip_path):
            files.download(zip_path)